# CTAO IRF and Visibility Examples

This notebook demonstrates how to use the `CTAOIRFManager` and `CTAOVisibilityEstimator`
to work with CTAO Instrument Response Functions (IRFs) and compute annual visibility.

We will cover:
- Loading IRFs
- Inspecting metadata
- Computing visibility
- Generating summary tables

In [1]:
from astropy.coordinates import SkyCoord
import astropy.units as u

from feupy.irf import (
    CTAOIRFManager,
    CTAOVisibilityEstimator,
    make_ctao_visibility_table,
)

## Define Target Position

We define a sky position using `SkyCoord`.
Here we use a Crab-like position.

In [6]:
# Example: Centaurus A
target = SkyCoord(
    ra=201.3651 * u.deg,
    dec=-43.0191 * u.deg,
    frame="icrs"
)

## Compute Annual Visibility

We compute the annual visibility for a given target position.

Constraints included:
- Astronomical night (Sun altitude)
- Moon avoidance

In [18]:
estimator = CTAOVisibilityEstimator(
    target=target,
    year=2025,
    time_step_min=30,
)

visibility_south = estimator.compute_visibility("cta_south")
visibility_south

cta_south: 100%|██████████████████████████████| 365/365 [00:16<00:00, 22.46it/s]


{'20': np.float64(345.5), '40': np.float64(336.5), '60': np.float64(319.0)}

## Compare CTAO North and South

In [16]:
vis_south = estimator.compute_visibility("cta_south")
vis_north = estimator.compute_visibility("cta_north")

print("South:", vis_south)
print("North:", vis_north)

cta_north: 100%|██████████████████████████████| 365/365 [00:17<00:00, 21.09it/s]

South: {'20': <Quantity 314.54749576>, '40': <Quantity 253.81603101>, '60': <Quantity 155.44372327>}
North: {'20': <Quantity 0.>, '40': <Quantity 0.>, '60': <Quantity 0.>}


## Airmass-Weighted Visibility

Optionally, visibility can be weighted by airmass.

In [17]:
estimator_weighted = CTAOVisibilityEstimator(
    target=target,
    year=2025,
    time_step_min=30,
    use_airmass_weight=True,
)

vis_weighted = estimator_weighted.compute_visibility("cta_south")

vis_weighted

cta_south: 100%|██████████████████████████████| 365/365 [00:16<00:00, 21.51it/s]


{'20': <Quantity 318.27553421>,
 '40': <Quantity 257.39943693>,
 '60': <Quantity 158.42077593>}

## Create Visibility Table

We generate a summary table for both CTAO sites.

In [20]:
df = make_ctao_visibility_table(estimator)

df

cta_north: 100%|██████████████████████████████| 365/365 [00:17<00:00, 20.66it/s]


,Observatory,Zenith (deg),Visibility (hours)
3,CTAO North,20,0.0
4,CTAO North,40,0.0
5,CTAO North,60,0.0
0,CTAO South,20,345.5
1,CTAO South,40,336.5
2,CTAO South,60,319.0


## Save Results

The table can be saved to CSV or LaTeX format.

In [21]:
make_ctao_visibility_table(estimator, save_path="visibility.csv")

cta_north: 100%|██████████████████████████████| 365/365 [00:17<00:00, 20.70it/s]


,Observatory,Zenith (deg),Visibility (hours)
3,CTAO North,20,0.0
4,CTAO North,40,0.0
5,CTAO North,60,0.0
0,CTAO South,20,345.5
1,CTAO South,40,336.5
2,CTAO South,60,319.0


## Interpretation

- The visibility is given in hours per year.
- Each bin corresponds to a zenith angle range:
  - 20° → low zenith (best performance)
  - 60° → higher zenith (worse sensitivity)

These results can be used to:
- Select optimal observation strategies
- Weight simulations
- Estimate realistic exposure

## Next Steps

- Combine visibility with IRFs for realistic simulations
- Integrate into CTAOAnalysis workflows
- Perform sensitivity studies